# SSD (Single Shot MultiBox Detector) для обнаружения объектов

**Модель:** SSD300 с backbone ResNet50

**Особенности архитектуры:**
- Одноэтапный детектор - предсказание за один проход
- Использует anchor boxes (фиксированные якорные боксы)
- Мультимасштабное обнаружение на разных слоях сети
- ResNet50 backbone для извлечения признаков
- Предсказания на 6 разных масштабах

**Преимущества:**
- Хороший баланс скорости и точности
- Отлично работает с объектами разных размеров
- Проще, чем Faster R-CNN
- Встроен в torchvision

**Недостатки:**
- Менее точный на мелких объектах, чем Faster R-CNN
- Медленнее, чем YOLO
- Требует тщательной настройки anchor boxes

**Главные отличия:**
- **От Faster R-CNN:** одноэтапный (без RPN), использует фиксированные anchor boxes
- **От YOLO:** использует anchor boxes (YOLOv8 - anchor-free), предсказывает на нескольких feature maps
- **От обоих:** уникальная архитектура с предсказаниями на разных слоях сети

In [ ]:
# Установка необходимых библиотек
!pip install torch torchvision matplotlib pillow numpy -q

In [ ]:
# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

print('Google Drive подключен успешно!')

In [ ]:
import torch
import torchvision
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict

print(f'PyTorch версия: {torch.__version__}')
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA устройство: {torch.cuda.get_device_name(0)}')

In [ ]:
# Конфигурация путей к данным
DATA_ROOT = '/content/drive/MyDrive/Датасет'  # Измените на свой путь
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, 'train/images')
TRAIN_LABEL_DIR = os.path.join(DATA_ROOT, 'train/labels')
VAL_IMG_DIR = os.path.join(DATA_ROOT, 'val/images')
VAL_LABEL_DIR = os.path.join(DATA_ROOT, 'val/labels')
TEST_IMG_DIR = os.path.join(DATA_ROOT, 'test/images')
TEST_LABEL_DIR = os.path.join(DATA_ROOT, 'test/labels')

# Гиперпараметры
BATCH_SIZE = 4  # SSD требует больше памяти, чем YOLO
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

print(f'Используемое устройство: {DEVICE}')

In [ ]:
# ============================================================
# АВТОМАТИЧЕСКОЕ ОПРЕДЕЛЕНИЕ КОЛИЧЕСТВА КЛАССОВ
# ============================================================
# Сканируем все файлы аннотаций и находим максимальный class_id
# NUM_CLASSES = max(class_id) + 1 (для самого класса) + 1 (для фона)
# SSD использует ту же схему что и Faster R-CNN

print('Анализ датасета для определения NUM_CLASSES...')

all_classes = []

# Сканирование всех директорий с labels
for label_dir in [TRAIN_LABEL_DIR, VAL_LABEL_DIR, TEST_LABEL_DIR]:
    if os.path.exists(label_dir):
        for label_file in os.listdir(label_dir):
            if label_file.endswith('.txt'):
                label_path = os.path.join(label_dir, label_file)
                with open(label_path, 'r') as f:
                    for line in f.readlines():
                        if line.strip():  # Пропускаем пустые строки
                            class_id = int(float(line.strip().split()[0]))
                            all_classes.append(class_id)

# Определение NUM_CLASSES
if len(all_classes) > 0:
    max_class_id = max(all_classes)
    NUM_CLASSES = max_class_id + 2  # +1 для индексации (т.к. классы начинаются с 0), +1 для фона
    unique_classes = sorted(set(all_classes))
    
    print(f'✅ Найденные классы в датасете: {unique_classes}')
    print(f'✅ Максимальный class_id: {max_class_id}')
    print(f'✅ NUM_CLASSES автоматически установлен на: {NUM_CLASSES}')
    print(f'   (формула: max_class_id + 2 = {max_class_id} + 2 = {NUM_CLASSES})')
    print(f'   {len(unique_classes)} классов объектов + 1 фон = {NUM_CLASSES} всего')
else:
    # Если не нашли ни одного класса, устанавливаем значение по умолчанию
    NUM_CLASSES = 2
    print(f'⚠️ Не найдено ни одного класса в аннотациях!')
    print(f'⚠️ NUM_CLASSES установлен на значение по умолчанию: {NUM_CLASSES}')

In [ ]:
class YOLODataset(Dataset):
    """Датасет для загрузки данных в формате YOLO"""
    
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith('.png')]
    
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        # Загрузка изображения
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        img = Image.open(img_path).convert('RGB')
        
        # Загрузка аннотаций (YOLO формат: class x_center y_center width height)
        label_path = os.path.join(self.label_dir, self.imgs[idx].replace('.png', '.txt'))
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    
                    # Конвертация из YOLO в формат SSD (такой же как R-CNN)
                    img_width, img_height = img.size
                    x_min = (x_center - width / 2) * img_width
                    y_min = (y_center - height / 2) * img_height
                    x_max = (x_center + width / 2) * img_width
                    y_max = (y_center + height / 2) * img_height
                    
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(class_id) + 1)
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.tensor([])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': image_id,
            'area': area,
            'iscrowd': iscrowd
        }
        
        img = torchvision.transforms.ToTensor()(img)
        
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
# Создание датасетов и загрузчиков данных
train_dataset = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR)
val_dataset = YOLODataset(VAL_IMG_DIR, VAL_LABEL_DIR)
test_dataset = YOLODataset(TEST_IMG_DIR, TEST_LABEL_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Размер обучающего набора: {len(train_dataset)}')
print(f'Размер валидационного набора: {len(val_dataset)}')
print(f'Размер тестового набора: {len(test_dataset)}')

In [ ]:
# ============================================================
# СОЗДАНИЕ МОДЕЛИ SSD (Single Shot MultiBox Detector)
# ============================================================
#
# SSD архитектура состоит из:
# 1. Base network (VGG16 или ResNet) - извлечение признаков
# 2. Auxiliary feature layers - дополнительные слои для мультимасштабности
# 3. Classification heads - предсказание классов на каждом масштабе
# 4. Regression heads - предсказание bbox на каждом масштабе
#
# Ключевая особенность SSD:
# - Предсказания делаются на РАЗНЫХ слоях сети (6 масштабов)
# - Каждый слой отвечает за объекты определенного размера:
#   * Ранние слои → мелкие объекты
#   * Поздние слои → крупные объекты
# - Использует anchor boxes (default boxes) разных размеров и соотношений
#
# Отличия от других моделей:
# ┌─────────────────┬──────────────────┬───────────────────┬─────────────────┐
# │                 │ Faster R-CNN     │ YOLOv8            │ SSD             │
# ├─────────────────┼──────────────────┼───────────────────┼─────────────────┤
# │ Этапы           │ 2 (RPN + Head)   │ 1 (direct)        │ 1 (direct)      │
# │ Anchor boxes    │ Да (RPN)         │ Нет (anchor-free) │ Да (multi-scale)│
# │ Предсказания    │ ROI pooling      │ Grid cells        │ Feature maps    │
# │ Масштабы        │ FPN (4 уровня)   │ 3 detection heads │ 6 feature maps  │
# │ Скорость        │ Медленная        │ Очень быстрая     │ Средняя         │
# └─────────────────┴──────────────────┴───────────────────┴─────────────────┘
# ============================================================

# Очистка кэша GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Загрузка предобученной модели SSD300 с VGG16 backbone
model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)

# Получение количества входных признаков в classification head
in_channels = [512, 1024, 512, 256, 256, 256]  # для каждого из 6 feature maps

# Создание новой classification head под наше количество классов
num_anchors = model.anchor_generator.num_anchors_per_location()
model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=NUM_CLASSES
)

# Перемещение модели на GPU/CPU
model = model.to(DEVICE)

print(f'Модель SSD300 создана и перемещена на {DEVICE}')
print(f'Количество классов (включая фон): {NUM_CLASSES}')
print(f'Количество anchor boxes на location: {num_anchors}')
print(f'Всего feature maps для предсказаний: {len(in_channels)}')

In [ ]:
# Настройка оптимизатора и планировщика
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LEARNING_RATE, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

print('Оптимизатор и планировщик настроены')

In [ ]:
# Функции обучения и валидации (такие же как у Faster R-CNN)
def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    total_loss = 0
    
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # SSD также возвращает словарь лоссов при обучении
        # Основные лоссы: classification loss + bbox regression loss
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
    
    return total_loss / len(data_loader)

def validate(model, data_loader, device):
    model.train()
    total_loss = 0
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()
    
    return total_loss / len(data_loader)

In [ ]:
# Обучение модели
print('Начало обучения SSD...')
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Обучение
    train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE)
    train_losses.append(train_loss)
    
    # Валидация
    val_loss = validate(model, val_loader, DEVICE)
    val_losses.append(val_loss)
    
    # Обновление learning rate
    lr_scheduler.step()
    
    print(f'Эпоха {epoch+1}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'LR: {optimizer.param_groups[0]["lr"]:.6f}')

print('Обучение завершено!')

In [ ]:
# Визуализация графиков обучения
fig, axes = plt.subplots(1, 1, figsize=(12, 6))

axes.plot(train_losses, label='Train Loss', marker='o', color='blue')
axes.plot(val_losses, label='Val Loss', marker='s', color='red')
axes.set_xlabel('Эпоха', fontsize=12)
axes.set_ylabel('Loss', fontsize=12)
axes.set_title('График обучения SSD', fontsize=14, fontweight='bold')
axes.legend(fontsize=11)
axes.grid(True, alpha=0.3)

axes.text(0.02, 0.98, 
    '💡 Как интерпретировать:\n'
    '✅ ХОРОШО: Обе линии снижаются, близки друг к другу\n'
    '⚠️ ПЕРЕОБУЧЕНИЕ: Синяя падает, красная растет\n'
    '⚠️ НЕДООБУЧЕНИЕ: Обе линии высокие, не снижаются\n'
    '📊 Меньше = Лучше',
    transform=axes.transAxes, fontsize=10,
    verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print(f'Финальный Train Loss: {train_losses[-1]:.4f}')
print(f'Финальный Val Loss: {val_losses[-1]:.4f}')

In [ ]:
# Функции для вычисления метрик (такие же как у Faster R-CNN)
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

def calculate_metrics(model, data_loader, device, iou_threshold=0.5, score_threshold=0.1):
    model.eval()
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)
            predictions = model(images)
            
            for pred, target in zip(predictions, targets):
                pred_boxes = pred['boxes'][pred['scores'] > score_threshold].cpu().numpy()
                pred_labels = pred['labels'][pred['scores'] > score_threshold].cpu().numpy()
                
                gt_boxes = target['boxes'].cpu().numpy()
                gt_labels = target['labels'].cpu().numpy()
                
                matched_gt = set()
                for pred_box, pred_label in zip(pred_boxes, pred_labels):
                    match_found = False
                    for i, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
                        if i not in matched_gt and pred_label == gt_label:
                            iou = compute_iou(pred_box, gt_box)
                            if iou >= iou_threshold:
                                true_positives += 1
                                matched_gt.add(i)
                                match_found = True
                                break
                    if not match_found:
                        false_positives += 1
                
                false_negatives += len(gt_boxes) - len(matched_gt)
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives
    }

In [ ]:
# Вычисление метрик на тестовом наборе
print('Вычисление метрик на тестовом наборе...')
test_metrics = calculate_metrics(model, test_loader, DEVICE, iou_threshold=0.5, score_threshold=0.1)

print('\n' + '='*50)
print('МЕТРИКИ НА ТЕСТОВОМ НАБОРЕ (SSD)')
print('='*50)
print(f'Precision: {test_metrics["precision"]:.4f}')
print(f'Recall: {test_metrics["recall"]:.4f}')
print(f'F1-Score: {test_metrics["f1_score"]:.4f}')
print(f'\nДетали:')
print(f'True Positives: {test_metrics["true_positives"]}')
print(f'False Positives: {test_metrics["false_positives"]}')
print(f'False Negatives: {test_metrics["false_negatives"]}')
print('='*50)

print('\n💡 Интерпретация метрик:')
print('Precision (точность) - какой % предсказаний правильный (больше = лучше)')
print('Recall (полнота) - какой % объектов найден (больше = лучше)')
print('F1-Score - баланс между точностью и полнотой (больше = лучше)')

In [ ]:
# Визуализация метрик
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

metrics_names = ['Precision', 'Recall', 'F1-Score']
metrics_values = [test_metrics['precision'], test_metrics['recall'], test_metrics['f1_score']]
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Значение', fontsize=12)
ax.set_title('Метрики качества SSD на тестовом наборе', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

for bar, value in zip(bars, metrics_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.text(0.5, 0.95, '📊 Больше = Лучше (максимум = 1.0)',
        transform=ax.transAxes, fontsize=11,
        ha='center', va='top',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация предсказаний на тестовых изображениях
model.eval()

indices = np.random.choice(len(test_dataset), min(9, len(test_dataset)), replace=False)

fig, axes = plt.subplots(3, 3, figsize=(20, 20))
axes = axes.flatten()

with torch.no_grad():
    for idx, ax in zip(indices, axes):
        img, target = test_dataset[idx]
        prediction = model([img.to(DEVICE)])[0]
        
        img_np = img.permute(1, 2, 0).cpu().numpy()
        
        ax.imshow(img_np)
        ax.axis('off')
        
        # Ground Truth (зеленый)
        for box, label in zip(target['boxes'], target['labels']):
            x1, y1, x2, y2 = box.cpu().numpy()
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor='green',
                                     facecolor='none', label='GT')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f'GT: {label.item()-1}',
                   color='green', fontsize=10, weight='bold',
                   bbox=dict(facecolor='white', alpha=0.7))
        
        # Предсказания (красный)
        for box, label, score in zip(prediction['boxes'], prediction['labels'], prediction['scores']):
            if score > 0.1:
                x1, y1, x2, y2 = box.cpu().numpy()
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                         linewidth=2, edgecolor='red',
                                         facecolor='none', linestyle='--')
                ax.add_patch(rect)
                ax.text(x2, y2+5, f'Pred: {label.item()-1} ({score:.2f})',
                       color='red', fontsize=10, weight='bold',
                       bbox=dict(facecolor='white', alpha=0.7))
        
        ax.set_title(f'Изображение {idx}', fontsize=12)

plt.suptitle('Результаты обнаружения SSD\n(Зеленый=Ground Truth, Красный=Предсказание)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Визуализация завершена!')

## Заключение

**SSD (Single Shot MultiBox Detector)** - это одноэтапный детектор с мультимасштабными предсказаниями:

**Когда использовать:**
- Когда нужен баланс между точностью и скоростью
- Для обнаружения объектов разных размеров
- Когда важна простота интеграции (torchvision)
- Для приложений с умеренными требованиями к скорости

**Когда НЕ использовать:**
- Для real-time приложений (используйте YOLO)
- Когда критична максимальная точность (используйте Faster R-CNN)
- При ограниченной GPU памяти

**Сравнение всех трех моделей:**

| Характеристика | Faster R-CNN | YOLOv8 | SSD |
|----------------|--------------|--------|-----|
| **Скорость** | 🐌 Медленная | 🚀 Очень быстрая | ⚡ Средняя |
| **Точность** | 🎯 Высокая | 👍 Хорошая | 👌 Средняя |
| **Память GPU** | 💾 Много | 💾 Мало | 💾 Средне |
| **Простота** | 🔧 Сложная | ✨ Очень простая | 👍 Простая |
| **Anchor boxes** | ✅ Да (RPN) | ❌ Нет (anchor-free) | ✅ Да (multi-scale) |
| **Этапы** | 2 этапа | 1 этап | 1 этап |
| **Лучше для** | Точность | Real-time | Баланс |

**Выбор модели:**
- 🎯 Нужна максимальная точность → **Faster R-CNN**
- ⚡ Нужна максимальная скорость → **YOLOv8**
- ⚖️ Нужен баланс → **SSD**